In [7]:
# %%
!pip install onnx onnx_tf
# %%
!pip install tensorflow-addons
# %%

In [11]:
import onnx
import torch
import torch.nn as nn
from torchvision import models

In [12]:
IMG_SIZE = 224
MODEL_PATH = "/content/waste_classifier_with_DenseNet201.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
# FineTune Model Function: DenseNet201
def get_model():
    model = models.densenet201(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False  # Prevents the pre-trained weights from being updated during training.

    # Replace classifier
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Linear(num_ftrs, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, 2)  # 2 classes: O, R
    )

    return model.to(DEVICE)

In [14]:
def load_trained_model(model_path):
    model = get_model()
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    return model

In [17]:
model = load_trained_model(MODEL_PATH)

# (Batch Size=1 , Channels=3, Height=224, Width=224)
dummy_input = torch.randn(1, 3, 224, 224)

# Move the dummy input to the same device as the model
dummy_input = dummy_input.to(DEVICE)

# Export the model to ONNX
torch.onnx.export(
    model,
    dummy_input,
    "waste_classifier_with_DenseNet201.onnx",
    export_params=True,        # Store the trained parameter weights inside the model file
    opset_version=11,          # A commonly supported ONNX opset version
    do_constant_folding=True,  # Whether to execute constant folding for optimization
    input_names=['input'],     # Name for the input tensor
    output_names=['output'],   # Name for the output tensor
)

print("PyTorch model successfully exported to waste_classifier_with_DenseNet201.onnx")

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet201_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet201_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


PyTorch model successfully exported to waste_classifier_with_DenseNet201.onnx


In [18]:
# Download the model
from google.colab import files
files.download('waste_classifier_with_DenseNet201.onnx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>